# Demo 2 · 函数链重排（XGBoost 业务排序）

复现官方 `function-chain-rerank`：同一个查询出两份榜单——**向量序**（纯 COSINE）和**业务序**（向量召回后，在 Milvus 内部跑 L0 函数链：流行度/价格亲和/新鲜度 → XGBoost 打分）。

In [ ]:
%pip install -q pymilvus==3.0.2 requests xgboost minio

## 1. 下载数据

In [ ]:
import json, requests

BASE = ("https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/"
        "demos/function-chain-rerank/data/synthetic-commerce-catalog-r1/")
products = requests.get(BASE + "products.json", timeout=120).json()["products"]
queries = requests.get(BASE + "queries.json", timeout=60).json()["queries"]
training = requests.get(BASE + "training-records.json", timeout=60).json()
print("商品:", len(products), "| 查询:", len(queries), "| 训练记录:", len(training["training_records"]))
print("特征顺序:", training["feature_order"])

## 2. 训练 XGBoost：5 个特征，单调递增约束

训练记录每条是 `features[5] → label(业务分 0~1)`。单调约束 `(1,1,1,1,1)` 保证每个特征只朝"更好"的方向加分，模型学不歪。

In [ ]:
import numpy as np, xgboost as xgb, os

recs = training["training_records"]
X = np.asarray([r["features"] for r in recs], dtype=np.float32)
y = np.asarray([r["label"] for r in recs], dtype=np.float32)
params = {"max_depth": 2, "eta": 0.08, "objective": "reg:squarederror",
          "eval_metric": "rmse", "nthread": 1, "seed": 42, "subsample": 1.0,
          "colsample_bytree": 1.0, "tree_method": "hist",
          "monotone_constraints": "(1,1,1,1,1)"}
model = xgb.train(params, xgb.DMatrix(X, label=y), num_boost_round=128)

os.makedirs("tmp", exist_ok=True)
UBJ = "tmp/xgb-reranker.ubj"
model.save_model(UBJ)
print("UBJ 字节数:", os.path.getsize(UBJ), "（官方 = 115265）")

## 3. 上传 MinIO，注册成 FileResource

XGBoost 算子从对象存储读模型；`path` 填 MinIO 里的相对路径。

In [ ]:
from minio import Minio

mc = Minio("127.0.0.1:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
BUCKET, OBJ = "a-bucket", "files/7_5A/xgb_reranker.ubj"
if not mc.bucket_exists(BUCKET):
    mc.make_bucket(BUCKET)
mc.fput_object(BUCKET, OBJ, UBJ, content_type="application/octet-stream")
print("上传成功:", mc.stat_object(BUCKET, OBJ).size, "bytes")
RES = "demo2_xgb_reranker"


## 4. 建集合，插入 240 个商品

In [ ]:
from pymilvus import MilvusClient, DataType, FunctionChain, FunctionChainStage
from pymilvus.function_chain import col, fn

COLL = "demo2_products"
client = MilvusClient(uri="http://127.0.0.1:19530")
if client.has_collection(COLL):
    client.drop_collection(COLL)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("id", DataType.INT64, is_primary=True)
schema.add_field("item_id", DataType.VARCHAR, max_length=32)
schema.add_field("title", DataType.VARCHAR, max_length=256)
schema.add_field("product_type", DataType.VARCHAR, max_length=64)
schema.add_field("brand", DataType.VARCHAR, max_length=128)
schema.add_field("description", DataType.VARCHAR, max_length=4096)
schema.add_field("display_price_usd", DataType.FLOAT)
schema.add_field("rating_value", DataType.FLOAT)
schema.add_field("clicks_30d", DataType.INT64)
schema.add_field("sales_30d", DataType.INT64)
schema.add_field("release_epoch", DataType.INT64)
schema.add_field("rating", DataType.FLOAT)
schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=1024)

ip = client.prepare_index_params()
ip.add_index(field_name="embedding", index_type="HNSW", metric_type="COSINE",
             params={"M": 16, "efConstruction": 64})
client.create_collection(COLL, schema=schema, index_params=ip)

KEEP = ["id", "item_id", "title", "product_type", "brand", "description",
        "display_price_usd", "rating_value", "clicks_30d", "sales_30d",
        "release_epoch", "rating"]
rows = [{k: p.get(k) for k in KEEP} | {"embedding": p["embedding"]} for p in products]
client.insert(COLL, data=rows)
client.flush(COLL)
client.load_collection(COLL)
print(client.get_collection_stats(COLL))

client.add_file_resource(name=RES, path=OBJ)
print(client.list_file_resources())

## 5. 一条 L0 函数链，五个算子

顺序执行：流行度（点击/销量加权）→ 价格亲和（线性衰减）→ 新鲜度（指数衰减）→ 语义分取 5 位小数 → **XGBoost 打出新的 `$score`**。下面的"业务序"就是链算完再排的榜。

In [ ]:
chain = (
    FunctionChain(FunctionChainStage.L0_RERANK, name="demo2_xgb_rerank")
    .map("popularity", fn.num_combine(col("clicks_30d"), col("sales_30d"),
                                     mode="weighted", weights=[0.3 / 12000, 0.7 / 850]))
    .map("price_affinity", fn.decay(col("display_price_usd"), function="linear",
                                    origin=0.0, scale=500.0, offset=0.0, decay=0.5))
    .map("freshness", fn.decay(col("release_epoch"), function="exp",
                               origin=1787184000, scale=180 * 86400, offset=0.0, decay=0.5))
    .map("normalized_semantic_score", fn.round_decimal(col("$score"), decimal=5))
    .map("$score", fn.xgboost(col("normalized_semantic_score"), col("rating"),
                              col("popularity"), col("price_affinity"), col("freshness"),
                              model_resource=RES, output="default"))
)
print("chain ready")

## 6. 一条查询，两份榜单

In [ ]:
q = queries[0]
print("查询:", q["query_text"])

vec = client.search(COLL, data=[q["embedding"]], anns_field="embedding",
                    search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                    limit=10, output_fields=["title", "display_price_usd", "rating"])
biz = client.search(COLL, data=[q["embedding"]], anns_field="embedding",
                    search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                    limit=10, function_chains=chain,
                    output_fields=["title", "display_price_usd", "rating"])

print(f'{"向量序":<40s} {"业务序":<40s}')
for v, b in zip(vec[0], biz[0]):
    print(f'{v["distance"]:.4f} {v["entity"]["title"][:30]:32s} | {b["distance"]:.4f} {b["entity"]["title"][:30]}')

## 7. 全部 6 个查询看一遍：顺序变了吗

两张榜的分数不在一个量纲：向量序是 COSINE 相似度，业务序是 XGBoost 业务分（0~1）。下面看 top-10 的重合与错位。


In [ ]:
for q in queries:
    vec = client.search(COLL, data=[q["embedding"]], anns_field="embedding",
                        search_params={"metric_type": "COSINE", "params": {"ef": 64}}, limit=10)
    biz = client.search(COLL, data=[q["embedding"]], anns_field="embedding",
                        search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                        limit=10, function_chains=chain)
    v_ids = [h["id"] for h in vec[0]]
    b_ids = [h["id"] for h in biz[0]]
    moves = sum(abs(v_ids.index(i) - b_ids.index(i)) for i in set(v_ids) & set(b_ids))
    print(f'{q["id"]:36s} 重合={len(set(v_ids)&set(b_ids))}/10 位移和={moves:2d}')


1. **`$score` 被改写**：业务序的 distance 是 XGBoost 输出（0~1 业务分），和向量序的相似度不在一个量纲，两张榜只比较顺序；
2. **链在 Milvus 内部执行**：五个算子一条链，按顺序算完只返回最终排序，客户端不做二次排序；
3. **单调约束保证方向正确**：价格只做折扣，点击/销量只加分，模型怎么训都不会把「更贵更旧」排前面。

## 清理


In [ ]:
client.drop_collection(COLL)
for fr in client.list_file_resources():
    if RES in str(fr):
        client.remove_file_resource(name=RES)
print("done")